# 🧲 Soumission par **NMI** (T1 → T2) — dataset1 & dataset3

Appariement *query* (T1) ↔ *target* (T2) par **Normalized Mutual Information** sur des volumes
ré-échantillonnés à **64³** (interpolation trilinéaire PyTorch).

**dataset2 est volontairement ignoré** (pas de fuite spatiale exploitable → NMI peu fiable).
La soumission ne contient donc que dataset1 + dataset3 (val + test).


## 1. Imports & configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Chemin local des données
DATA_ROOT = r"C:\Users\User\PycharmProjects\hack_paris\data\ehl-paris-medical-image-retrieval"
VOL_SIZE  = 64
print("DATA_ROOT =", DATA_ROOT)
print("existe ? ", os.path.isdir(DATA_ROOT))


## 2. Chargement des volumes (resize 64³ trilinéaire)

In [ ]:
def resolve(path):
    full = os.path.join(DATA_ROOT, path)
    if not os.path.exists(full) and full.endswith(".gz"):
        full = full[:-3]
    return full

def load_vol(path, size=VOL_SIZE):
    vol = nib.load(resolve(path)).get_fdata().astype(np.float32)
    t = torch.tensor(vol).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(size, size, size), mode="trilinear", align_corners=False)
    return t.squeeze().numpy()


## 3. Score NMI (Normalized Mutual Information)

In [ ]:
def nmi_score(v1, v2, bins=32):
    v1, v2 = v1.flatten(), v2.flatten()
    hist2d, _, _ = np.histogram2d(v1, v2, bins=bins)
    pxy = hist2d / (hist2d.sum() + 1e-12)
    px, py = pxy.sum(1), pxy.sum(0)
    px_py = px[:, None] * py[None, :]
    mask = pxy > 0
    mi = (pxy[mask] * np.log(pxy[mask] / (px_py[mask] + 1e-12))).sum()
    hx = -(px[px > 0] * np.log(px[px > 0])).sum()
    hy = -(py[py > 0] * np.log(py[py > 0])).sum()
    return 2 * mi / (hx + hy + 1e-8)


## 4. Classement d'un split par NMI

In [ ]:
def rank_nmi(df_q, df_g, desc=""):
    gallery_vols, gids = [], []
    for _, row in tqdm(df_g.iterrows(), total=len(df_g), desc=f"  gallery {desc}"):
        gallery_vols.append(load_vol(row["target_image"]))
        gids.append(row["target_id"])
    results = []
    for _, row in tqdm(df_q.iterrows(), total=len(df_q), desc=f"  queries {desc}"):
        vq = load_vol(row["query_image"])
        scores = np.array([nmi_score(vq, vg) for vg in gallery_vols])
        order  = np.argsort(scores)[::-1]
        results.append((row["query_id"], [gids[j] for j in order]))
    return results


## 5. Construction de la soumission (dataset1 + dataset3, ds2 ignoré)

In [ ]:
def build_submission(out_path=r"C:\Users\User\PycharmProjects\hack_paris\submission_nmi_d1_d3.csv"):
    rows = []
    for dataset_id in ["dataset1", "dataset2", "dataset3"]:
        for split in ["val", "test"]:
            qcsv = os.path.join(DATA_ROOT, dataset_id, f"{split}_queries.csv")
            gcsv = os.path.join(DATA_ROOT, dataset_id, f"{split}_gallery.csv")
            if not os.path.exists(qcsv):
                continue
            if dataset_id == "dataset2":
                print(f"\n=== {dataset_id}/{split} → IGNORÉ ===")
                continue
            print(f"\n=== {dataset_id}/{split} ===")
            df_q = pd.read_csv(qcsv)
            df_g = pd.read_csv(gcsv)
            results = rank_nmi(df_q, df_g, desc=f"({dataset_id}/{split})")
            for qid, ranked in results:
                rows.append({"query_id": qid, "target_id_ranking": " ".join(ranked)})
    df = pd.DataFrame(rows)
    df.to_csv(out_path, index=False)
    print(f"\n✓ {out_path} : {len(df)} lignes")
    return df


## 6. Exécution

In [ ]:
df = build_submission(r"C:\Users\User\PycharmProjects\hack_paris\submission_nmi_d1_d3.csv")
df.head()


---
**Note.** dataset2 étant ignoré, la soumission a **237 lignes** (ds1 : 40+100, ds3 : 20+77).
Selon le challenge, les query de dataset2 absentes peuvent être notées 0 — pour une soumission
complète, fusionne ce fichier avec un classement ds2 (cf. `pixelwise_retrieval.ipynb`).
